In [3]:
import json
import random
import uuid
import time
from datetime import datetime, timezone
from faker import Faker

# Initialize Faker
fake = Faker()

# Pre-define realistic product categories and devices for our bias
DEVICES = ["mobile", "desktop", "tablet"]
OS_MAP = {"mobile": ["iOS", "Android"], "desktop": ["Windows", "macOS", "Linux"], "tablet": ["iPadOS", "Android"]}
BROWSERS = ["Safari", "Chrome", "Firefox", "Edge"]
CATEGORIES = {
    "Electronics": ["Audio", "Computers", "Accessories"],
    "Apparel": ["Men's", "Women's", "Shoes"],
    "Home": ["Furniture", "Decor", "Kitchen"]
}

# Your custom bias logic
CATEGORY_WEIGHTS = {"desktop": [0.60, 0.10, 0.30], "mobile": [0.10, 0.60, 0.30], "tablet": [0.30, 0.30, 0.40]}

# --- THE FUNNEL STATE DICTIONARY ---
# This will temporarily hold users who are actively clicking through the site
active_sessions = {}

def generate_ecommerce_event():
    event_id = str(uuid.uuid4())
    timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

    # 1. Decide: Progress an existing active session, or start a new one?
    # We give a 40% chance to progress an existing session (if any exist) to simulate concurrent traffic
    if active_sessions and random.random() < 0.40:
        
        # Pick a random active user session
        session_id = random.choice(list(active_sessions.keys()))
        session_data = active_sessions[session_id]
        
        # Pull their exact user and product data from memory to maintain the sequence
        user = session_data["user"]
        product = session_data["product"]
        event_type = session_data["next_step"]
        
        # Determine what happens NEXT in their funnel
        if event_type == "add_to_cart":
            # Decide quantity now that they are adding to cart
            product["quantity"] = random.randint(1, 3) 
            
            # Give them a 30% chance to eventually buy, otherwise they abandon the cart
            if random.random() < 0.30:
                active_sessions[session_id]["next_step"] = "purchase"
            else:
                del active_sessions[session_id] # Cart abandoned, remove from memory
                
        elif event_type == "purchase":
            # The funnel is complete! Remove them from active memory.
            del active_sessions[session_id]
            
    else:
        # 2. Start a completely NEW session
        session_id = str(uuid.uuid4())
        event_type = "view_item" # New sessions ALWAYS start with a view
        
        # Build the User Object
        device_type = random.choice(DEVICES)
        user = {
            "user_id": f"usr_{random.randint(10000, 99999)}",
            "name": fake.name(),
            "email": fake.email(),
            "location": {
                "country": "United States",
                "state": fake.state(),
                "city": fake.city()
            },
            "device": {
                "type": device_type,
                "os": random.choice(OS_MAP[device_type]),
                "browser": random.choice(BROWSERS)
            }
        }
        
        # Build the Product Object based on device bias
        category = random.choices(list(CATEGORIES.keys()), weights=CATEGORY_WEIGHTS[device_type], k=1)[0]
        sub_category = random.choice(CATEGORIES[category])
        product = {
            "product_id": f"prod_{random.randint(1000, 9999)}",
            "title": fake.catch_phrase(),
            "category": category,
            "sub_category": sub_category,
            "price": round(random.uniform(9.99, 499.99), 2),
            "quantity": 1 # Default view quantity
        }
        
        # Will this new user continue down the funnel later?
        # Let's give a 50% chance they eventually add this item to their cart
        if random.random() < 0.50:
            active_sessions[session_id] = {
                "user": user,
                "product": product,
                "next_step": "add_to_cart" # Queue up the next step
            }

    # 3. Assemble the final JSON payload
    payload = {
        "event_id": event_id,
        "session_id": session_id,
        "timestamp": timestamp,
        "event_type": event_type,
        "user": user,
        "product": product
    }
    
    return payload

# --- Execution Block ---
from kafka import KafkaProducer

# ... [Keep all your bias logic, dictionaries, and generate_ecommerce_event() function here] ...

# --- NEW Execution Block ---
if __name__ == "__main__":
    print("Connecting to Kafka and starting data stream... (Press Ctrl+C to stop)\n")
    
    try:
        # Initialize the Kafka Producer
        # For now, we point it to 'localhost:9092' assuming a local broker setup
        producer = KafkaProducer(
            bootstrap_servers=['localhost:9092'],
            value_serializer=lambda v: json.dumps(v).encode('utf-8')
        )
    except Exception as e:
        print(f"Failed to connect to Kafka broker. Is it running? Error: {e}")
        exit(1)
        
    try:
        for _ in range(1000):
            event_data = generate_ecommerce_event()
            
            # Send the JSON payload securely to the Kafka topic
            producer.send('ecommerce_clickstream', event_data)
            
            # Print a confirmation to the terminal so you know it's working
            print(f"Produced event {event_data['event_id']} [{event_data['event_type']}] to Kafka.")
            
            # Pause briefly to simulate real-time traffic
            time.sleep(0.5)
            
    except KeyboardInterrupt:
        print("\nData stream stopped.")
    finally:
        # Ensure all queued messages are sent before shutting down
        producer.flush()
        producer.close()
        print("Kafka Producer closed.")

Connecting to Kafka and starting data stream... (Press Ctrl+C to stop)

Kafka Producer closed.


KafkaTimeoutError: KafkaTimeoutError: Failed to update metadata after 60.0 secs.